# Tabular Time Series Related API Demo with NOAA Weather Data

Copyright (c) Microsoft Corporation. All rights reserved. <br>
Licensed under the MIT License.

In this notebook, you will learn how to use the Tabular Time Series related API to filter the data by time windows for sample data uploaded to Azure blob storage. 

The detailed APIs to be demoed in this script are:
- Create Tabular Dataset instance
- Assign fine timestamp column and coarse timestamp column for Tabular Dataset to activate Time Series related APIs
- Clear fine timestamp column and coarse timestamp column
- Filter in data before a specific time
- Filter in data after a specific time
- Filter in data in a specific time range
- Filter in data for recent time range

Besides above APIs, you'll also see:
- Create and load a Workspace
- Load National Oceanic & Atmospheric (NOAA) weather data into Azure blob storage
- Create and register NOAA weather data as a Tabular dataset
- Re-load Tabular Dataset from your Workspace

## Import Dependencies

If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, run the cells below to install the Azure Machine Learning Python SDK and create an Azure ML Workspace that's required for this demo.

## Prepare Environment

Print out your version of the Azure ML Python SDK. Version 1.0.60 or above is required for TabularDataset with timeseries attribute. 

In [1]:
import azureml.core
azureml.data.__version__

ImportError: libffi-ae16d830.so.6.0.4: cannot open shared object file: No such file or directory

## Import Packages

In [2]:
# import packages
import os

import pandas as pd
import azureml.core
import azureml.data

from calendar import monthrange
from datetime import datetime, timedelta

from azureml.core import Dataset, Datastore, Workspace, Run
from azureml.data.azure_storage_datastore import AzureFileDatastore, AzureBlobDatastore
from azureml.data.dataset_factory import TabularDatasetFactory
from azureml.opendatasets import NoaaIsdWeather

Failure while loading azureml_run_type_providers. Failed to load entrypoint hyperdrive = azureml.train.hyperdrive:HyperDriveRun._from_run_dto with exception (azureml-core 1.0.60 (/home/cody/.local/lib/python3.7/site-packages), Requirement.parse('azureml-core==1.0.57.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.PipelineRun = azureml.pipeline.core:PipelineRun._from_dto with exception (azureml-core 1.0.60 (/home/cody/.local/lib/python3.7/site-packages), Requirement.parse('azureml-core==1.0.57.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.ReusedStepRun = azureml.pipeline.core:StepRun._from_reused_dto with exception (azureml-core 1.0.60 (/home/cody/.local/lib/python3.7/site-packages), Requirement.parse('azureml-core==1.0.57.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.StepRun = azureml.pipeline.core:StepRun._from_dto with exception (azureml-core 1.0.60 (/home

## Set up Configuraton and Create Azure ML Workspace

If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, go through the [configuration notebook](https://github.com/Azure/MachineLearningNotebooks/blob/master/configuration.ipynb) first if you haven't already to establish your connection to the Azure ML Workspace.

In [3]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

print(ws.name, ws.resource_group, ws.location, ws.subscription_id, dstore.name, sep = '\n')

copetersTest
copetersRG
eastus
60582a10-b9fd-49f1-a546-c4194134bba8
workspaceblobstore


<font color=red>Remove below codes and eanble above codes before releasing it.</font>

## Load Data to Blob Storage

This demo uses public NOAA weather data. You can replace this data with your own. The first cell below creates a Pandas Dataframe object with the first 6 months of 2019 NOAA weather data. The last cell saves the data to a CSV file and uploads the CSV file to Azure blob storage to the location specified in the datapath variable. Currently, the Dataset class only reads uploaded files from blob storage. 

**NOTE:** the below code loads in NOAA ISD weather data from all of 2019. This will take ~15 minutes. To reduce the size of data, we will only keep specific rows with a given stationName.

In [27]:
target_years = [2019]

for year in target_years:
    for month in range(1, 12+1):
        path = 'data/{}/{:02d}/'.format(year, month)
        
        try:  
            start = datetime(year, month, 1)
            end   = datetime(year, month, monthrange(year, month)[1]) + timedelta(days=1)
            isd   = NoaaIsdWeather(start, end).to_pandas_dataframe()
            isd   = isd[isd['stationName'].str.contains('FLORIDA', regex=True, na=False)]
            
            os.makedirs(path, exist_ok=True)
            isd.to_parquet(path + 'data.parquet')
        except Exception as e:
            print('Month {} in year {} likely has no data.\n'.format(month, year))
            print('Exception: {}'.format(e))

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=1/', '/year=2019/month=2/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=1/part-00043-tid-6736193421618840939-fc84afa8-e440-4d19-8ad1-ffd7f9d4ec76-55342.c000.snappy.parquet under container isdweatherdatacontainer
Reading ISDWeather/year=2019/month=2/part-00174-tid-6736193421618840939-fc84afa8-e440-4d19-8ad1-ffd7f9d4ec76-55473.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=71535.37 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=71617.88 [ms]
ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=2/', '/year=2019/month=3/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/m

Upload data to blob storage.

In [7]:
dstore.upload('data', 'noaa-isd-data', overwrite=True, show_progress=True)

Uploading an estimated of 8 files
Uploading data/2019/01/data.parquet
Uploading data/2019/02/data.parquet
Uploading data/2019/03/data.parquet
Uploading data/2019/04/data.parquet
Uploading data/2019/05/data.parquet
Uploading data/2019/06/data.parquet
Uploading data/2019/07/data.parquet
Uploading data/2019/08/data.parquet
Uploaded data/2019/01/data.parquet, 1 files out of an estimated total of 8
Uploaded data/2019/07/data.parquet, 2 files out of an estimated total of 8
Uploaded data/2019/06/data.parquet, 3 files out of an estimated total of 8
Uploaded data/2019/08/data.parquet, 4 files out of an estimated total of 8
Uploaded data/2019/05/data.parquet, 5 files out of an estimated total of 8
Uploaded data/2019/04/data.parquet, 6 files out of an estimated total of 8
Uploaded data/2019/03/data.parquet, 7 files out of an estimated total of 8
Uploaded data/2019/02/data.parquet, 8 files out of an estimated total of 8
Uploaded 8 files


$AZUREML_DATAREFERENCE_a191099d704a4d13a923592c6a780ab7

## Create & Register Tabular Dataset with time-series trait from Blob

The API on Tabular datasets with time-series trait is specially designed to handle Tabular time-series data and time related operations more efficiently. By registering your time-series dataset, you are publishing your dataset to your workspace so that it is accessible to anyone with the same subscription id. 

Create Tabular Dataset instance from blob storage datapath

In [9]:
datastore_path = [(dstore, 'noaa-isd-data/*/*/data.parquet')]
dataset        = Dataset.Tabular.from_parquet_files(path=datastore_path, partition_format = 'noaa-isd-data/{coarse_time:yyyy/MM}/data.parquet')

Assign fine timestamp column for Tabular Dataset to activate Time Series related APIs. The column to be assigned should be a Date type, otherwise the assigning will fail.

In [10]:
tsd = dataset.with_timestamp_columns(fine_grain_timestamp='datetime', coarse_grain_timestamp='coarse_time')

You can convert the entire dataset to a Pandas dataframe, or filter on the timestamp column before converting. Datasets are lazily evaluated.

In [11]:
df  = tsd.to_pandas_dataframe()
df.head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,865450,99999,2019-01-01 10:00:00,-34.067,-56.238,68.0,60.0,4.1,20.0,1012.4,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050032,2019-01-01
1,865450,99999,2019-01-01 11:00:00,-34.067,-56.238,68.0,40.0,5.1,22.2,1013.0,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050033,2019-01-01
2,865450,99999,2019-01-01 12:00:00,-34.067,-56.238,68.0,40.0,6.2,24.0,1012.8,...,9999.0,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050034,2019-01-01
3,865450,99999,2019-01-01 13:00:00,-34.067,-56.238,68.0,30.0,5.7,25.7,1012.2,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050035,2019-01-01
4,865450,99999,2019-01-01 14:00:00,-34.067,-56.238,68.0,30.0,5.1,28.7,1011.5,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050036,2019-01-01


Register the dataset for easy access from anywhere in Azure ML and keep track of versions, lineage. 

In [12]:
# register dataset to Workspace
registered_ds = tsd.register(ws, 'noaa-isd-florida', description='Data for Tabular Dataset with time-series trait demo.', tags={ 'type': 'TabularDataset' })

## Reload the Dataset from Workspace

In [13]:
# get dataset by dataset name
tsd = Dataset.get_by_name(ws, name='noaa-dataset')
tsd.to_pandas_dataframe().head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
0,865450,99999,2019-01-01 10:00:00,-34.067,-56.238,68.0,60.0,4.1,20.0,1012.4,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050032
1,865450,99999,2019-01-01 11:00:00,-34.067,-56.238,68.0,40.0,5.1,22.2,1013.0,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050033
2,865450,99999,2019-01-01 12:00:00,-34.067,-56.238,68.0,40.0,6.2,24.0,1012.8,...,24.0,9999.0,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050034
3,865450,99999,2019-01-01 13:00:00,-34.067,-56.238,68.0,30.0,5.7,25.7,1012.2,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050035
4,865450,99999,2019-01-01 14:00:00,-34.067,-56.238,68.0,30.0,5.1,28.7,1011.5,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,1.0,4050036


## Filter Data by Time Windows

Once your data has been loaded into the notebook, you can query by time using the time_before(), time_after(), time_between(), and time_recent() functions. You can also choose to drop or keep certain columns.  

### Before Time Input

In [16]:
# select data that occurs before a specified date
tsd2 = tsd.time_before(datetime(2019, 6, 12))
tsd2.to_pandas_dataframe().sort_values(by='datetime').tail(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
20306,854880,99999,2019-06-11 23:00:00,-29.916,-71.200,147.0,230.0,1.5,13.0,NaN,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,11,1.0,7197284
19566,720735,73805,2019-06-11 23:02:00,30.349,-85.788,21.0,120.0,4.6,28.3,NaN,...,NaN,NaN,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,11,1.0,1907673
19567,720735,73805,2019-06-11 23:38:00,30.349,-85.788,21.0,160.0,3.1,24.4,NaN,...,1.0,40.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,11,1.0,1907674
19568,720735,73805,2019-06-11 23:53:00,30.349,-85.788,21.0,120.0,4.6,24.4,1013.5,...,6.0,51.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,11,1.0,1907675
19918,722108,12894,2019-06-11 23:53:00,26.536,-81.755,9.0,230.0,4.6,32.2,1014.6,...,1.0,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,2062366


## After Time Input

In [17]:
# select data that occurs after a specified date
tsd2 = tsd.time_after(datetime(2019, 6, 1))
tsd2.to_pandas_dataframe().sort_values(by='datetime').tail(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
11576,722108,12894,2019-08-29 04:59:00,26.536,-81.755,9.0,NaN,NaN,NaN,NaN,...,24.0,69.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943678
11577,722108,12894,2019-08-29 05:53:00,26.536,-81.755,9.0,0.0,0.0,27.2,1012.4,...,NaN,NaN,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943679
11578,722108,12894,2019-08-29 06:50:00,26.536,-81.755,9.0,0.0,0.0,27.0,NaN,...,NaN,NaN,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943680
11579,722108,12894,2019-08-29 06:53:00,26.536,-81.755,9.0,350.0,1.5,27.2,1011.9,...,NaN,NaN,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943681
11580,722108,12894,2019-08-29 07:01:00,26.536,-81.755,9.0,330.0,1.5,27.2,NaN,...,NaN,NaN,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943682


## Before & After Time Inputs

You can chain time functions together.

In [18]:
# select data that occurs within a given time range
tsd2 = tsd.time_after(datetime(2019, 1, 6)).time_before(datetime(2019, 1, 12))
tsd2.to_pandas_dataframe().sort_values(by='datetime').tail(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
412,722108,12894,2019-01-11 22:53:00,26.536,-81.755,9.0,60.0,2.1,20.6,1019.8,...,1.0,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,5395512
556,013170,99999,2019-01-11 23:00:00,60.383,5.333,36.0,130.0,3.0,2.5,1013.8,...,1.0,0.0,NaN,BERGEN/FLORIDA,NO,013170-99999,2019,11,1.0,8676413
745,854880,99999,2019-01-11 23:00:00,-29.916,-71.200,147.0,280.0,4.6,17.0,NaN,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,11,1.0,9461643
257,720735,73805,2019-01-11 23:53:00,30.349,-85.788,21.0,0.0,0.0,8.9,1023.8,...,1.0,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,11,1.0,5225735
413,722108,12894,2019-01-11 23:53:00,26.536,-81.755,9.0,80.0,3.1,19.4,1020.4,...,1.0,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,5395513


### Time Range Input

In [19]:
# another way to select data that occurs within a given time range
tsd2 = tsd.time_between(start_time=datetime(2019, 1, 5, 23, 59, 59), end_time=datetime(2019, 1, 7))
tsd2.to_pandas_dataframe().head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
0,865450,99999,2019-01-06 10:00:00,-34.067,-56.238,68.0,30.0,3.6,19.6,1010.0,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,6,1.0,4050077
1,865450,99999,2019-01-06 11:00:00,-34.067,-56.238,68.0,20.0,6.7,22.2,1009.7,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,6,1.0,4050078
2,865450,99999,2019-01-06 12:00:00,-34.067,-56.238,68.0,10.0,6.2,24.7,1009.6,...,24.0,9999.0,NaN,FLORIDA,UY,865450-99999,2019,6,1.0,4050079
3,865450,99999,2019-01-06 13:00:00,-34.067,-56.238,68.0,20.0,5.1,27.0,1008.8,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,6,1.0,4050080
4,865450,99999,2019-01-06 14:00:00,-34.067,-56.238,68.0,20.0,6.2,28.7,1008.5,...,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,6,1.0,4050081


## Time Recent Input

This function takes in a datetime.timedelta and returns a dataset containing the data from datetime.now()-timedelta() to datetime.now().

In [20]:
tsd2 = tsd.time_recent(timedelta(weeks=4, days=0))
tsd2.to_pandas_dataframe().head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
0,854880,99999,2019-08-02 12:00:00,-29.917,-71.2,146.0,130.0,3.6,6.8,1019.9,...,24.0,0.0,NaN,LA FLORIDA,CI,854880-99999,2019,2,1.0,2033038
1,854880,99999,2019-08-02 12:00:00,-29.916,-71.2,147.0,130.0,3.6,7.0,NaN,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,2,1.0,2033039
2,854880,99999,2019-08-02 13:00:00,-29.916,-71.2,147.0,130.0,1.5,10.0,NaN,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,2,1.0,2033040
3,854880,99999,2019-08-02 14:00:00,-29.916,-71.2,147.0,200.0,1.5,13.0,NaN,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,2,1.0,2033041
4,854880,99999,2019-08-02 15:00:00,-29.917,-71.2,146.0,100.0,1.5,14.9,1019.8,...,NaN,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,2,1.0,2033042


**NOTE:** This will return an empty dataframe there is no data within the last 2 days.

In [21]:
tsd2 = tsd.time_recent(timedelta(days=2))
tsd2.to_pandas_dataframe().tail(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__
71,722108,12894,2019-08-29 04:59:00,26.536,-81.755,9.0,NaN,NaN,NaN,NaN,...,24.0,69.0,None,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943678
72,722108,12894,2019-08-29 05:53:00,26.536,-81.755,9.0,0.0,0.0,27.2,1012.4,...,NaN,NaN,None,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943679
73,722108,12894,2019-08-29 06:50:00,26.536,-81.755,9.0,0.0,0.0,27.0,NaN,...,NaN,NaN,None,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943680
74,722108,12894,2019-08-29 06:53:00,26.536,-81.755,9.0,350.0,1.5,27.2,1011.9,...,NaN,NaN,None,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943681
75,722108,12894,2019-08-29 07:01:00,26.536,-81.755,9.0,330.0,1.5,27.2,NaN,...,NaN,NaN,None,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,29,1.0,7943682


### Drop Columns

<font color=red>The columns to be dropped should NOT include timstamp columns.</font><br>Below operation will lead to exception.

In [22]:
try:
    tsd2 = tsd.drop_columns(columns=['snowDepth', 'version', 'datetime'])
except Exception as e:
    print('Expected exception : {}'.format(str(e)))

2019-08-30 18:33:14.140225 | ActivityCompleted: Activity=drop_columns, HowEnded=Failure, Duration=0.22 [ms], Info = {'activity_id': 'cb7fe997-31cf-4091-98b9-7a2308a7eb00', 'activity_name': 'drop_columns', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.60', 'completionStatus': 'Success', 'durationMs': 4802.28}, Exception=ValueError; Column datetime should not be dropped because it's a timestamp column. Please exclude it or use with_timestamp_columns with value None to reset it before dropping columns.


Expected exception : Column datetime should not be dropped because it's a timestamp column. Please exclude it or use with_timestamp_columns with value None to reset it before dropping columns.


Drop will succeed if modify column list to exclude timestamp columns.

In [23]:
tsd2 = tsd.drop_columns(columns=['snowDepth', 'version', 'upload_date'])
tsd2.take(5).to_pandas_dataframe().sort_values(by='datetime')

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,presentWeatherIndicator,pastWeatherIndicator,precipTime,precipDepth,stationName,countryOrRegion,p_k,year,day,__index_level_0__
0,865450,99999,2019-01-01 10:00:00,-34.067,-56.238,68.0,60.0,4.1,20.0,1012.4,...,0.0,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,4050032
1,865450,99999,2019-01-01 11:00:00,-34.067,-56.238,68.0,40.0,5.1,22.2,1013.0,...,3.0,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,4050033
2,865450,99999,2019-01-01 12:00:00,-34.067,-56.238,68.0,40.0,6.2,24.0,1012.8,...,3.0,0.0,24.0,9999.0,FLORIDA,UY,865450-99999,2019,1,4050034
3,865450,99999,2019-01-01 13:00:00,-34.067,-56.238,68.0,30.0,5.7,25.7,1012.2,...,1.0,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,4050035
4,865450,99999,2019-01-01 14:00:00,-34.067,-56.238,68.0,30.0,5.1,28.7,1011.5,...,1.0,NaN,NaN,NaN,FLORIDA,UY,865450-99999,2019,1,4050036


### Keep Columns

<font color=red>The columns to be kept should ALWAYS include timstamp columns.</font><br>Below operation will lead to exception.

In [24]:
try:
    tsd2 = tsd.keep_columns(columns=['snowDepth'], validate=False)
except Exception as e:
    print('Expected exception : {}'.format(str(e)))

2019-08-30 18:33:15.148495 | ActivityCompleted: Activity=keep_columns, HowEnded=Failure, Duration=0.24 [ms], Info = {'activity_id': '69807446-8571-4535-b586-9ffcb5afdf11', 'activity_name': 'keep_columns', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.60', 'completionStatus': 'Success', 'durationMs': 960.45}, Exception=ValueError; Column datetime should be included because it's a timestamp column


Expected exception : Column datetime should be included because it's a timestamp column


Keep will succeed if modify column list to include timestamp columns.

In [25]:
tsd2 = tsd.keep_columns(columns=['snowDepth', 'datetime'], validate=False)
tsd2.take(5).to_pandas_dataframe().sort_values(by='datetime')

,datetime,snowDepth
0,2019-01-01 10:00:00,False
1,2019-01-01 11:00:00,False
2,2019-01-01 12:00:00,False
3,2019-01-01 13:00:00,False
4,2019-01-01 14:00:00,False


## Resetting Timestamp Columns

Rules for reseting are:
- You cannot assign 'None' to fine_grain_timestamp while assign a valid column name to coarse_grain_timestamp because coarse_grain_timestamp is optional while fine_grain_timestamp is mandatory for Tabular time series data.
- If you assign 'None' to fine_grain_timestamp, then both fine_grain_timestamp and coarse_grain_timestamp will all be cleared.
- If you assign only 'None' to coarse_grain_timestamp, then only coarse_grain_timestamp will be cleared.

In [26]:
# Illegal clearing, exception is expected.
try:
    tsd2 = tsd.with_timestamp_columns(fine_grain_timestamp=None, coarse_grain_timestamp='partition_date')
except Exception as e:
    print('Cleaning not allowed because {}'.format(str(e)))

# clear both
tsd2 = tsd.with_timestamp_columns(fine_grain_timestamp=None, coarse_grain_timestamp=None)
print('after clean both with None/None, timestamp columns are: {}'.format(tsd2.timestamp_columns))

# clear coarse_grain_timestamp only and assign 'datetime' as fine timestamp column
tsd2 = tsd2.with_timestamp_columns(fine_grain_timestamp='datetime', coarse_grain_timestamp=None)
print('after clean coarse timestamp column, timestamp columns are: {}'.format(tsd2.timestamp_columns))

2019-08-30 18:33:16.152460 | ActivityCompleted: Activity=with_timestamp_columns, HowEnded=Failure, Duration=0.08 [ms], Info = {'activity_id': 'ca41e542-3fc8-4b28-b697-52b4a366dff5', 'activity_name': 'with_timestamp_columns', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.60', 'completionStatus': 'Success', 'durationMs': 965.15}, Exception=ValueError; Cannot assign coarse_grain_timestamp separately while fine_grain_timestamp is None.


Cleaning not allowed because Cannot assign coarse_grain_timestamp separately while fine_grain_timestamp is None.
after clean both with None/None, timestamp columns are: (None, None)
after clean coarse timestamp column, timestamp columns are: ('datetime', None)


In [31]:
help(azureml.data.tabular_dataset)